# CoughScan: диагностика пропущенного кашля
Запускайте ячейки сверху вниз в Google Colab (обычная CPU-среда подойдёт).
Сервер и приложение не изменяются. Нужны исходные WAV/ZIP, `cough_detection_final_optimized.keras` и **парный** `cough_scaler_final_optimized.pkl` из того же развёртывания.

Не загружайте записи чужих разговоров без разрешения. Notebook обрабатывает файлы в вашей сессии Colab и загружает публичную YAMNet; серверный Python не запускается. Загружайте только свой доверенный scaler: joblib может выполнять код при десериализации.

**Цель:** отделить громкость/запись, входные признаки, модель и серверные множители. Числовые оценки модели не считаются калиброванной вероятностью или медицинским заключением.

In [ ]:
%pip -q install tensorflow-hub tf-keras librosa soundfile joblib scikit-learn pandas matplotlib ipywidgets

In [ ]:
import os, sys, json, logging, csv, zipfile, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import tensorflow as tf
import tensorflow_hub as hub
import sklearn, joblib
import matplotlib.pyplot as plt
from IPython.display import Audio, display
from google.colab import files
logger = logging.getLogger("cough_diagnostics")
print({"python":sys.version, "tensorflow":tf.__version__, "librosa":librosa.__version__, "sklearn":sklearn.__version__})
ROOT = Path('/content/cough_diagnostics'); ROOT.mkdir(exist_ok=True)
AUDIO = ROOT / 'audio'; AUDIO.mkdir(exist_ok=True)

## 1. Файлы
Сначала скачайте архив **сразу после короткого теста**:
- [Все доступные WAV одним ZIP](https://alexbrosoj-yam-134b.twc1.net/coughs/audio/all)
- [Метаданные записей](https://alexbrosoj-yam-134b.twc1.net/records/all?limit=500&include_audio=true)

Эти URL взяты из присланного кода; доступность сервера здесь не подтверждена. Название `coughs` вводит в заблуждение: текущий код сохраняет и выгружает также отрицательные результаты. ZIP включает только файлы, ещё связанные с записями БД. Записи БД старше часа удаляются при следующем upload/startup.

Загрузите ZIP/WAV и два файла модели. Можно выполнить ячейку повторно, чтобы добавить файлы. Если модель недоступна, можно выполнить базовую проверку YAMNet, но сравнения собственного классификатора не будет.

In [ ]:
uploaded = files.upload()
for name, data in uploaded.items():
    name = Path(name).name
    destination = ROOT / name
    destination.write_bytes(data)
    if name.lower().endswith('.wav'):
        (AUDIO / name).write_bytes(data)
    elif name.lower().endswith('.zip'):
        with zipfile.ZipFile(destination) as archive:
            total = sum(i.file_size for i in archive.infolist())
            if total > 1024**3: raise ValueError('ZIP больше 1 GiB после распаковки: выберите меньшую сессию')
            for i, info in enumerate(archive.infolist()):
                if not info.is_dir() and info.filename.lower().endswith('.wav'):
                    (AUDIO / f'{destination.stem}_{i}_{Path(info.filename).name}').write_bytes(archive.read(info))
print('WAV:', len(list(AUDIO.glob('*.wav'))))
print('Артефакты:', [p.name for p in ROOT.iterdir() if p.is_file()])

In [ ]:
# Необязательно: скачать пару артефактов из публичного GitHub вместо ручной загрузки.
# Включите только если на сервере используются именно эти версии.
DOWNLOAD_GITHUB_ARTIFACTS = True
if DOWNLOAD_GITHUB_ARTIFACTS:
    import requests
    for name in ['cough_detection_final_optimized.keras','cough_scaler_final_optimized.pkl']:
        url = 'https://raw.githubusercontent.com/AlexBrosOJ/Yam/main/' + name
        destination = ROOT / name
        with requests.get(url, stream=True, timeout=(15,180)) as response:
            response.raise_for_status()
            with destination.open('wb') as output:
                for block in response.iter_content(1024*1024): output.write(block)
        if destination.read_bytes()[:100].startswith(b'version https://git-lfs.github.com'):
            raise ValueError('Это указатель Git LFS, загрузите реальный артефакт')
        print(name, destination.stat().st_size)


In [ ]:
YAMNET_MODEL = hub.load('https://tfhub.dev/google/yamnet/1')
class_map = YAMNET_MODEL.class_map_path().numpy().decode()
with open(class_map) as f:
    CLASS_NAMES = [row['display_name'] for row in csv.DictReader(f)]
COUGH_INDEX = CLASS_NAMES.index('Cough') if 'Cough' in CLASS_NAMES else CLASS_NAMES.index('Coughing')
print('Класс кашля:', CLASS_NAMES[COUGH_INDEX], COUGH_INDEX)
MODEL_PATH = ROOT / 'cough_detection_final_optimized.keras'
SCALER_PATH = ROOT / 'cough_scaler_final_optimized.pkl'
OUR_MODEL = None; SCALER = None
if MODEL_PATH.exists() and SCALER_PATH.exists():
    # Не переключаем Keras/не меняем scaler молча при несовместимости.
    OUR_MODEL = tf.keras.models.load_model(MODEL_PATH, compile=False)
    SCALER = joblib.load(SCALER_PATH)
    print('Model:', OUR_MODEL.input_shape, OUR_MODEL.output_shape)
    print('Scaler:', type(SCALER).__name__, getattr(SCALER, 'n_features_in_', None))
    for p in [MODEL_PATH, SCALER_PATH]:
        print(p.name, hashlib.sha256(p.read_bytes()).hexdigest())
else:
    raise FileNotFoundError('Нет модели/scaler. Выполните предыдущую ячейку скачивания или загрузите оба файла с сервера, затем повторите эту ячейку.')

## 2. Исходные серверные функции
Следующая ячейка содержит **без изменения** две функции из присланного кода: извлечение 2079 признаков и `fast_enhanced_check`. Это воспроизведение, а не предлагаемое исправление. Обучающая семантика выхода (1 = кашель?) должна быть подтверждена обучающим кодом.

In [ ]:
def extract_features_new(waveform, sr, yamnet_model):
    """Извлекает РОВНО 2079 фич как в обученной модели"""
    try:
        if np.max(np.abs(waveform)) < 0.01:
            return None

        max_val = np.max(np.abs(waveform))
        waveform = waveform / (max_val + 1e-8)

        waveform_tf = tf.convert_to_tensor(waveform, dtype=tf.float32)
        _, embeddings, _ = yamnet_model(waveform_tf)

        avg_pool = tf.reduce_mean(embeddings, axis=0).numpy()
        max_pool = tf.reduce_max(embeddings, axis=0).numpy()

        mfcc = librosa.feature.mfcc(y=waveform, sr=sr, n_mfcc=13, hop_length=512)
        mfcc_mean = np.mean(mfcc, axis=1)
        mfcc_std = np.std(mfcc, axis=1)

        spectral_centroid = librosa.feature.spectral_centroid(y=waveform, sr=sr, hop_length=512)[0]
        spectral_bandwidth = librosa.feature.spectral_bandwidth(y=waveform, sr=sr, hop_length=512)[0]

        spectral_features = np.array([
            np.mean(spectral_centroid),
            np.std(spectral_centroid),  
            np.mean(spectral_bandwidth)
        ])

        zcr = librosa.feature.zero_crossing_rate(waveform, hop_length=512)[0]
        zcr_mean = np.mean(zcr)

        rms = librosa.feature.rms(y=waveform, hop_length=512)[0]
        rms_mean = np.mean(rms)

        combined = np.concatenate([
            avg_pool,
            max_pool,  
            mfcc_mean,
            mfcc_std,
            spectral_features,
            [zcr_mean, rms_mean]
        ])

        expected_dim = 2079
        if len(combined) != expected_dim:
            logger.warning(f"Размерность фич {len(combined)} вместо {expected_dim}")
            if len(combined) > expected_dim:
                combined = combined[:expected_dim]
            else:
                padding = np.zeros(expected_dim - len(combined))
                combined = np.concatenate([combined, padding])

        return combined

    except Exception as e:
        logger.error(f"Ошибка извлечения фич: {e}")
        return None

def fast_enhanced_check(waveform, sr, original_prob):
    """Быстрая улучшенная проверка (умеренные штрафы)"""

    modified_prob = original_prob

    # 1. Проверка соотношения низких/средних частот
    mfcc = librosa.feature.mfcc(y=waveform, sr=sr, n_mfcc=13, hop_length=512)
    low_freq = np.mean(np.abs(mfcc[:4, :]))      # Низкие
    mid_freq = np.mean(np.abs(mfcc[4:8, :]))     # Средние
    low_to_mid = low_freq / (mid_freq + 1e-6)

    # Умеренные штрафы: макс -20%
    if low_to_mid < 0.18:  # Слишком мало низких (ложное)
        modified_prob *= 0.78  # -10% (было -35%)
    elif low_to_mid > 0.27:  # Много низких (кашель)
        modified_prob *= 1.25  # +10% (было +20%)

    # 2. Проверка резкости атаки
    envelope = np.abs(waveform)
    envelope_smooth = np.convolve(envelope, np.ones(1000)/1000, mode='same')

    from scipy.signal import find_peaks
    peaks, properties = find_peaks(envelope_smooth, height=np.max(envelope_smooth)*0.3)

    if len(peaks) > 0:
        peak_to_rms_ratio = envelope_smooth[peaks[0]] / (np.sqrt(np.mean(waveform**2)) + 1e-6)

        # Умеренные штрафы
        if peak_to_rms_ratio < 7.1:  # Низкая пиковость (ложное)
            modified_prob *= 0.82  # -10% (было -20%)
        elif peak_to_rms_ratio > 7.3:  # Высокая пиковость (кашель)
            modified_prob *= 1.25  # +10% (было +15%)
    else:
        # Нет пиков - легкий штраф
        modified_prob *= 0.75  # -5% (было -30%)

    # 3. Проверка спектральной плоскости
    spectral_flatness = np.mean(librosa.feature.spectral_flatness(y=waveform))

    # Умеренные штрафы
    if spectral_flatness < 0.04:  # Слишком тонально (ложное)
        modified_prob *= 0.82  # -5% (было -15%)
    elif spectral_flatness > 0.05:  # Более шумно (кашель)
        modified_prob *= 1.152  # +5% (было +10%)

    # 4. Количество пиков
    zcr = librosa.feature.zero_crossing_rate(waveform, hop_length=256)[0]
    zcr_peaks = len(find_peaks(zcr, height=np.mean(zcr)*1.5)[0])

    # Умеренные штрафы
    if zcr_peaks < 35:  # Мало изменений (ложное)
        modified_prob *= 0.9  # -5% (было -25%)
    elif zcr_peaks > 45:  # Много изменений (кашель)
        modified_prob *= 1.1  # +5% (было +15%)

    # Возвращаем float, а не numpy тип!
    return float(np.clip(modified_prob, 0.0, 1.0))

In [ ]:
def diagnose(path):
    info = sf.info(path)
    waveform, sr = librosa.load(path, sr=16000, mono=True)  # явно resample, не подмена sample rate
    if len(waveform) == 0 or not np.isfinite(waveform).all():
        raise ValueError('Пустое аудио или NaN/Inf')
    first = waveform[:16000]  # сервер отбрасывает всё после первой секунды
    peak = float(np.max(np.abs(first)))
    rms = float(np.sqrt(np.mean(first**2)))
    row = dict(file=path.name, sha256=hashlib.sha256(path.read_bytes()).hexdigest(),
               source_sr=info.samplerate, channels=info.channels, duration_s=info.duration,
               rms=rms, rms_dbfs=20*np.log10(max(rms,1e-12)), peak=peak,
               would_skip_rms=rms<0.02, would_skip_peak=peak<0.01,
               original=np.nan, enhanced=np.nan, server_detected=np.nan)
    # Контрольная YAMNet на всём оригинальном ресемплированном сигнале, без peak-normalization.
    scores, _, _ = YAMNET_MODEL(tf.convert_to_tensor(waveform, dtype=tf.float32))
    scores = scores.numpy()
    row['yamnet_cough_max'] = float(scores[:,COUGH_INDEX].max())
    row['yamnet_cough_mean'] = float(scores[:,COUGH_INDEX].mean())
    means = scores.mean(axis=0)
    row['yamnet_top5'] = '; '.join(f'{CLASS_NAMES[i]}:{means[i]:.3f}' for i in np.argsort(means)[-5:][::-1])
    if OUR_MODEL is not None:
        normalized = first / (peak + 1e-8)
        normalized = np.pad(normalized, (0, max(0,16000-len(normalized))))[:16000]
        if peak > 0:
            features = extract_features_new(normalized, sr, YAMNET_MODEL)
            if features is None: raise ValueError('Ошибка извлечения признаков')
            if features.shape != (2079,) or not np.isfinite(features).all():
                raise ValueError(f'Неверные признаки: {features.shape}')
            if getattr(SCALER,'n_features_in_',2079) != 2079:
                raise ValueError('Scaler ожидает другое число признаков')
            scaled = SCALER.transform(features.reshape(1,-1))
            prediction = np.asarray(OUR_MODEL.predict(scaled,verbose=0))
            if prediction.shape != (1,1):
                raise ValueError(f'Неожиданный выход модели {prediction.shape}: проверьте обучающий код')
            original = float(prediction[0,0])
            if not np.isfinite(original) or not 0<=original<=1:
                raise ValueError(f'Выход модели вне [0,1]: {original}')
            enhanced = fast_enhanced_check(normalized, sr, original)
            row.update(original=original, enhanced=enhanced,
                       multiplier=enhanced/original if original>0 else np.nan,
                       server_detected=bool(rms>=0.02 and peak>=0.01 and enhanced>0.75))
            # original/enhanced ниже RMS-порога — контрфактическая диагностика:
            # живой сервер вообще не вызвал бы для них модель.
    return row

paths = sorted(AUDIO.glob('*.wav'))
assert paths, 'Сначала загрузите WAV или ZIP'
rows = []
for i,path in enumerate(paths):
    try: rows.append(diagnose(path))
    except Exception as error: rows.append(dict(file=path.name,error=repr(error)))
    if (i+1)%10==0: print(f'{i+1}/{len(paths)}')
results = pd.DataFrame(rows)
results.to_csv(ROOT/'diagnostics.csv', index=False)
display(results)
print('server_detected вычисляется только при наличии собственной модели; NaN = нет результата, не «нет кашля».')
if 'would_skip_rms' in results:
    print('RMS-фильтр отсеял бы', int(results.would_skip_rms.fillna(False).sum()), 'из', len(results), 'файлов.')

## 3. Прослушивание — обязательный шаг
Выберите файлы с отчётливым кашлем и фон. `normalize=False`: плеер не поднимает тихую запись автоматически. На фактическую громкость также влияют динамики/наушники. Не склеивайте разные телефоны и сессии: `device_id=ios_iPhone` не уникален.

In [ ]:
INDEX = 0  # меняйте индекс файла
path = paths[INDEX]
y, sr = librosa.load(path, sr=None, mono=True)
print(path.name, sr, 'Hz', len(y)/sr, 'сек')
display(Audio(y, rate=sr, normalize=False))
plt.figure(figsize=(12,3)); plt.plot(np.arange(len(y))/sr, y)
plt.xlabel('Секунды'); plt.ylabel('Амплитуда'); plt.ylim(-1,1); plt.grid(); plt.show()
display(results[results.file==path.name])

## 4. Разметка кнопками: слушайте и нажимайте
Ничего заполнять в Excel и загружать обратно не нужно. `label` — только ваш ответ: слышен кашель или нет.

Запустите ячейку ниже. Она показывает по одному фрагменту: **Да, кашель / Нет кашля / Не уверен**. Ответ сохраняется автоматически, появляется следующий фрагмент. Модельные оценки скрыты, чтобы не подсказывать ответ. Можно проверить сначала 10–15 файлов и остановиться.

Если слишком тихо, включите «Усилить только для прослушивания». WAV и вход модели не меняются. Источник можно оставить неизвестным. Кнопки «Назад/Вперёд» позволяют исправить ответы.

CSV скачивается **только по кнопке «Скачать ответы CSV»**. Это резервная копия; для сравнения в следующей ячейке загрузка CSV не требуется. До отключения Colab скачайте ответы: файлы сессии не постоянны.

Размечаем первую секунду каждого файла, которую видел сервер. Это метрики по фрагментам, а не точный подсчёт отдельных кашлей.


In [ ]:
# Paste this entire file into ONE new Colab code cell after the WAV/results cells.
from pathlib import Path
import html
import numpy as np
import pandas as pd
import librosa
import ipywidgets as widgets
from IPython.display import Audio, display, clear_output
from google.colab import files, output as colab_output

colab_output.enable_custom_widget_manager()
assert 'results' in globals() and 'AUDIO' in globals(), 'Сначала выполните загрузку WAV и анализ.'
ROOT = Path(ROOT)
LABEL_PATH = ROOT / 'labels.csv'
_audio_dir = Path(AUDIO)
_required = ['file'] + (['sha256'] if 'sha256' in results else [])
_labels = results[_required].drop_duplicates('file').copy()
_labels['label'] = np.nan
_labels['source'] = ''
_labels['reviewed'] = False
if LABEL_PATH.exists():
    _old = pd.read_csv(LABEL_PATH).fillna('')
    _old = _old.drop_duplicates('file', keep='last').set_index('file')
    for _i, _row in _labels.iterrows():
        if _row.file not in _old.index:
            continue
        _saved = _old.loc[_row.file]
        if 'sha256' in _saved and 'sha256' in _row and _saved.sha256 != _row.sha256:
            continue
        _value = pd.to_numeric(_saved.get('label', ''), errors='coerce')
        _labels.at[_i, 'label'] = _value if _value in (0, 1) else np.nan
        _labels.at[_i, 'source'] = _saved.get('source', '')
        _labels.at[_i, 'reviewed'] = (str(_saved.get('reviewed', '')).lower() == 'true' or _value in (0, 1))
_labels = _labels.reset_index(drop=True)
assert len(_labels), 'Нет файлов для прослушивания.'
_pending = _labels.index[~_labels.reviewed].tolist()
_position = _pending[0] if _pending else 0
_panel = widgets.Output()
_header = widgets.HTML()
_status = widgets.HTML()
_source = widgets.Dropdown(options=[('Источник неизвестен', ''), ('Живой кашель / микрофон', 'live'),
                                    ('Звук из видео / динамика', 'video'), ('Обычный фон', 'background')],
                           description='Источник:', style={'description_width':'initial'})
_louder = widgets.Checkbox(value=False, description='Усилить только для прослушивания')
_yes = widgets.Button(description='Да, кашель', button_style='success')
_no = widgets.Button(description='Нет кашля', button_style='danger')
_unsure = widgets.Button(description='Не уверен')
_back = widgets.Button(description='← Назад')
_next = widgets.Button(description='Вперёд →')
_export = widgets.Button(description='Скачать ответы CSV', button_style='info')


def _save_labels():
    temporary = LABEL_PATH.with_suffix('.tmp')
    _labels.to_csv(temporary, index=False)
    temporary.replace(LABEL_PATH)
    # Compatible with subsequent metrics cells; no upload dialog needed.
    globals()['labels'] = _labels.copy()


def _render_label():
    row = _labels.iloc[_position]
    done = int(_labels.reviewed.sum())
    value = 'Да' if row.label == 1 else 'Нет' if row.label == 0 else 'Не уверен' if row.reviewed else 'Ещё не отмечено'
    _header.value = (f'<b>Фрагмент {_position+1}/{len(_labels)} · проверено {done}/{len(_labels)}</b><br>'
                     f'{html.escape(row.file)}<br>Ответ: <b>{value}</b>')
    _source.value = row.source if row.source in ['', 'live', 'video', 'background'] else ''
    with _panel:
        clear_output(wait=True)
        try:
            y, sr = librosa.load(_audio_dir / row.file, sr=None, mono=True)
            # Label the exact interval seen by the server, even for longer uploaded files.
            y = y[:sr]
            print('Первая секунда записи. Оценки модели скрыты, чтобы не влиять на ответ.')
            if _louder.value:
                print('Включено усиление плеера. Исходный WAV и вход модели НЕ меняются.')
            display(Audio(y, rate=sr, normalize=_louder.value, autoplay=True))
            print('Если звук не включился автоматически, нажмите ▶ в плеере.')
        except Exception as error:
            print('Не удалось воспроизвести:', error, '— выберите «Не уверен».')
    _back.disabled = _position == 0
    _next.disabled = _position == len(_labels)-1


def _answer(value):
    global _position
    _labels.at[_position, 'label'] = value
    _labels.at[_position, 'source'] = _source.value
    _labels.at[_position, 'reviewed'] = True
    _save_labels()
    pending = _labels.index[~_labels.reviewed].tolist()
    later = [i for i in pending if i > _position]
    if later:
        _position = later[0]
    elif pending:
        _position = pending[0]
    else:
        _status.value = '<b>Готово! Все записи просмотрены. Нажмите «Скачать ответы CSV».</b>'
    _render_label()


def _move(offset):
    global _position
    _position = max(0, min(len(_labels)-1, _position+offset))
    _render_label()


def _download_answers(_):
    _save_labels()
    files.download(str(LABEL_PATH))

_yes.on_click(lambda _: _answer(1))
_no.on_click(lambda _: _answer(0))
_unsure.on_click(lambda _: _answer(np.nan))
_back.on_click(lambda _: _move(-1))
_next.on_click(lambda _: _move(1))
_export.on_click(_download_answers)
_louder.observe(lambda change: _render_label(), names='value')
display(widgets.VBox([_header, _source, _louder, _panel,
                      widgets.HBox([_yes, _no, _unsure]), widgets.HBox([_back, _next, _export]), _status]))
print('Ответ сохраняется после каждого нажатия в этой сессии Colab. Можно остановиться и продолжить.')
print('Перед закрытием/сбросом среды скачайте CSV: файлы среды Colab не постоянны.')
_render_label()


In [ ]:
assert 'labels' in globals(), 'Сначала отметьте записи кнопками выше'
assert labels.file.is_unique, 'В labels.csv повторяются имена файлов'
labeled = results.merge(labels,on='file')
labeled['label'] = pd.to_numeric(labeled.label,errors='coerce')
labeled = labeled[labeled.label.isin([0,1])].copy()
assert len(labeled), 'Нет размеченных строк'

def metrics(truth,pred):
    truth=np.asarray(truth,dtype=bool); pred=np.asarray(pred,dtype=bool)
    tp=int((truth & pred).sum()); fn=int((truth & ~pred).sum())
    fp=int((~truth & pred).sum()); tn=int((~truth & ~pred).sum())
    return dict(TP=tp,FN=fn,FP=fp,TN=tn,
                recall=tp/(tp+fn) if tp+fn else np.nan,
                precision=tp/(tp+fp) if tp+fp else np.nan,
                false_positive_rate=fp/(fp+tn) if fp+tn else np.nan)

comparisons=[]
for source, group in [('all',labeled)]+list(labeled.groupby('source')):
    for column in ['original','enhanced']:
        g=group.dropna(subset=[column])
        if not len(g): continue
        for threshold in [0.3,0.5,0.65,0.75,0.9]:
            pred=g[column]>threshold
            comparisons.append(dict(source=source,variant=column+'_without_RMS_gate',threshold=threshold,
                                    **metrics(g.label,pred)))
    g=group.dropna(subset=['enhanced'])
    if len(g): comparisons.append(dict(source=source,variant='current_server',threshold=0.75,
                                      **metrics(g.label,g.server_detected)))
comparison=pd.DataFrame(comparisons)
display(comparison)
comparison.to_csv(ROOT/'comparison.csv',index=False)
print('Не выбирайте финальный порог на этой же тестовой выборке. Нужны отдельная валидация и независимый тест.')

In [ ]:
with zipfile.ZipFile(ROOT/'diagnostic_results.zip','w',zipfile.ZIP_DEFLATED) as archive:
    for name in ['diagnostics.csv','comparison.csv','labels.csv']:
        path=ROOT/name
        if path.exists(): archive.write(path,name)
files.download(str(ROOT/'diagnostic_results.zip'))

## Что прислать после запуска
1. `diagnostics.csv` и заполненную разметку (без чужих личных разговоров).
2. Скрин Model input/output shape и Scaler n_features_in_.
3. Несколько строк серверного лога `orig=... enhanced=...` для соответствующих файлов.
4. Код обучения/формирования признаков и точные версии зависимостей сервера (`pip freeze`).

Размерность 2079 **не доказывает совпадение признаков с обучением**. Число, порядок, длительность, MFCC-параметры и scaler должны совпасть. Пока не меняйте порог/модель на продакшене по одному положительному примеру.